In [ ]:
import torch

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -U datasets huggingface_hub

In [ ]:
import datasets
import huggingface_hub

print(datasets.__version__)
print(huggingface_hub.__version__)

In [ ]:
# Core deep learning libraries
!pip install -q torch torchvision torchaudio

# Hugging Face ecosystem
!pip install -q transformers datasets accelerate evaluate

# Tokenization and preprocessing
!pip install -q sentencepiece tokenizers

# Machine learning utilities
!pip install -q scikit-learn pandas numpy

# Visualization libraries
!pip install -q matplotlib seaborn

# NLP utilities
!pip install -q nltk

# Emotion and keyword extraction support
!pip install -q keybert

# Model optimization
!pip install -q optimum

print("All dependencies installed successfully.")

In [ ]:
import os
import re
import torch
import random
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, f1_score
from keybert import KeyBERT

import matplotlib.pyplot as plt

print("Libraries imported successfully.")

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

print("Random seed initialized.")

In [ ]:
# Create project directories

project_dirs = [
    "data",
    "models",
    "checkpoints",
    "logs",
    "outputs"
]

for directory in project_dirs:
    os.makedirs(directory, exist_ok=True)

print("Project directories created successfully.")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
project_path = "/content/drive/MyDrive/MentalHealthAI"

os.makedirs(project_path, exist_ok=True)

print("Project folder ready:", project_path)

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

result = classifier("I feel happy today.")

print(result)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("google-research-datasets/go_emotions")

In [ ]:
dataset = load_dataset("go_emotions")

print(dataset)

In [ ]:
sample = dataset["train"][0]

print(sample)

In [ ]:
import transformers
import datasets

environment_info = {
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "datasets_version": datasets.__version__,
    "gpu_available": torch.cuda.is_available()
}

print(environment_info)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("google-research-datasets/go_emotions")

In [ ]:
from datasets import load_dataset

dataset = load_dataset("go_emotions")

print(dataset)

In [ ]:
print("Training Samples:", len(dataset["train"]))
print("Validation Samples:", len(dataset["validation"]))
print("Test Samples:", len(dataset["test"]))

In [ ]:
for i in range(5):
    print(dataset["train"][i])
    print("-" * 50)

In [ ]:
emotion_labels = [
    "admiration",
    "amusement",
    "anger",
    "annoyance",
    "approval",
    "caring",
    "confusion",
    "curiosity",
    "desire",
    "disappointment",
    "disapproval",
    "disgust",
    "embarrassment",
    "excitement",
    "fear",
    "gratitude",
    "grief",
    "joy",
    "love",
    "nervousness",
    "optimism",
    "pride",
    "realization",
    "relief",
    "remorse",
    "sadness",
    "surprise",
    "neutral"
]

In [ ]:
sample = dataset["train"][0]

print("Text:")
print(sample["text"])

print("\nEmotion Labels:")

for label_id in sample["labels"]:
    print(emotion_labels[label_id])

In [ ]:
for sample in dataset["train"]:
    if len(sample["labels"]) > 1:
        print("Text:")
        print(sample["text"])

        print("\nDetected Emotions:")

        for label_id in sample["labels"]:
            print("-", emotion_labels[label_id])

        break

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")

print("Tokenizer loaded successfully.")

In [ ]:
import re

def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
sample_text = "I am REALLY stressed today!!! Visit https://example.com"

cleaned = clean_text(sample_text)

print(cleaned)

In [ ]:
def preprocess_text(example):

    example["text"] = clean_text(example["text"])

    return example

In [ ]:
dataset = dataset.map(preprocess_text)

In [ ]:
print(dataset["train"][0]["text"])

In [ ]:
MAX_LENGTH = 128

def tokenize_function(example):

    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

In [ ]:
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

In [ ]:
print(tokenized_dataset["train"][0].keys())

In [ ]:
print(tokenized_dataset["train"][0]["input_ids"][:20])

In [ ]:
NUM_LABELS = len(emotion_labels)

def encode_labels(example):

    label_vector = [0.0] * NUM_LABELS

    for label in example["labels"]:
        label_vector[label] = 1.0

    example["labels"] = label_vector

    return example

In [ ]:
tokenized_dataset = tokenized_dataset.map(encode_labels)

In [ ]:
print(tokenized_dataset["train"][0]["labels"])

In [ ]:
tokenized_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

In [ ]:
sample = tokenized_dataset["train"][0]

print("Input IDs Shape:", sample["input_ids"].shape)
print("Attention Mask Shape:", sample["attention_mask"].shape)
print("Labels Shape:", sample["labels"].shape)

In [ ]:
tokenized_dataset.save_to_disk("data/go_emotions_processed")

In [ ]:
import numpy as np
import torch

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

In [ ]:
MODEL_NAME = "distilroberta-base"

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=28,
    problem_type="multi_label_classification"
)

print("Model loaded successfully.")

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("Using device:", device)

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = torch.sigmoid(
        torch.tensor(logits)
    ).numpy()

    predictions = (predictions >= 0.5).astype(int)

    f1 = f1_score(
        labels,
        predictions,
        average="micro"
    )

    precision = precision_score(
        labels,
        predictions,
        average="micro"
    )

    recall = recall_score(
        labels,
        predictions,
        average="micro"
    )

    return {
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="checkpoints",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_dir="logs",

    logging_steps=100,

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    fp16=torch.cuda.is_available(),

    report_to="none"
)

In [ ]:
import transformers

print(transformers.__version__)

In [ ]:
tokenized_dataset = tokenized_dataset.map(
    lambda example: {
        "labels": [float(x) for x in example["labels"]]
    }
)

In [ ]:
from datasets import Features, Sequence, Value

In [ ]:
features = tokenized_dataset["train"].features.copy()

features["labels"] = Sequence(
    Value("float32")
)

tokenized_dataset = tokenized_dataset.cast(features)

In [ ]:
tokenized_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

In [ ]:
sample = tokenized_dataset["train"][0]

print(sample["labels"])
print(sample["labels"].dtype)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate()

print(results)

In [ ]:
trainer.save_model("models/emotion_model")

tokenizer.save_pretrained(
    "models/emotion_model"
)

print("Model saved successfully.")

In [ ]:
import shutil

destination = "/content/drive/MyDrive/MentalHealthAI/emotion_model"

shutil.copytree(
    "models/emotion_model",
    destination,
    dirs_exist_ok=True
)

print("Model backed up to Google Drive.")

In [ ]:
from transformers import pipeline

emotion_classifier = pipeline(
    "text-classification",
    model="models/emotion_model",
    tokenizer=tokenizer,
    top_k=None
)

In [ ]:
text = "I feel exhausted and stressed because of work."

prediction = emotion_classifier(text)

print(prediction)

In [ ]:
id2label = {
    0: "admiration",
    1: "amusement",
    2: "anger",
    3: "annoyance",
    4: "approval",
    5: "caring",
    6: "confusion",
    7: "curiosity",
    8: "desire",
    9: "disappointment",
    10: "disapproval",
    11: "disgust",
    12: "embarrassment",
    13: "excitement",
    14: "fear",
    15: "gratitude",
    16: "grief",
    17: "joy",
    18: "love",
    19: "nervousness",
    20: "optimism",
    21: "pride",
    22: "realization",
    23: "relief",
    24: "remorse",
    25: "sadness",
    26: "surprise",
    27: "neutral"
}

In [ ]:
formatted_predictions = []

for item in prediction[0]:

    label_id = int(item["label"].split("_")[1])

    formatted_predictions.append({
        "emotion": id2label[label_id],
        "score": round(item["score"], 4)
    })

formatted_predictions

In [ ]:
filtered_predictions = [
    item for item in formatted_predictions
    if item["score"] > 0.10
]

filtered_predictions

In [ ]:
def analyze_emotion(text, threshold=0.10):

    prediction = emotion_classifier(text)

    formatted_predictions = []

    for item in prediction[0]:

        label_id = int(item["label"].split("_")[1])

        emotion = id2label[label_id]

        score = round(item["score"], 4)

        if score >= threshold:

            formatted_predictions.append({
                "emotion": emotion,
                "score": score
            })

    formatted_predictions = sorted(
        formatted_predictions,
        key=lambda x: x["score"],
        reverse=True
    )

    return {
        "text": text,
        "detected_emotions": formatted_predictions
    }

In [ ]:
result = analyze_emotion(
    "I feel mentally exhausted because of continuous work pressure and deadlines."
)

result

In [ ]:
def analyze_emotion(text, threshold=0.10):

    prediction = emotion_classifier(text)

    formatted_predictions = []

    for item in prediction[0]:

        label_id = int(item["label"].split("_")[1])

        emotion = id2label[label_id]

        score = round(item["score"], 4)

        if score >= threshold:

            formatted_predictions.append({
                "emotion": emotion,
                "score": score
            })

    formatted_predictions = sorted(
        formatted_predictions,
        key=lambda x: x["score"],
        reverse=True
    )

    dominant_emotion = (
        formatted_predictions[0]["emotion"]
        if formatted_predictions
        else "neutral"
    )

    return {
        "text": text,
        "dominant_emotion": dominant_emotion,
        "detected_emotions": formatted_predictions
    }

In [ ]:
result = analyze_emotion(
    "I feel lonely and emotionally drained."
)

result

In [ ]:
result = analyze_emotion(
    "I feel lonely and emotionally drained."
)

result

In [ ]:
positive_emotions = [
    "joy",
    "love",
    "gratitude",
    "optimism",
    "admiration",
    "approval",
    "caring",
    "excitement",
    "pride",
    "relief"
]

negative_emotions = [
    "sadness",
    "anger",
    "fear",
    "grief",
    "remorse",
    "disappointment",
    "disapproval",
    "disgust",
    "nervousness",
    "annoyance",
    "embarrassment"
]

In [ ]:
def get_mood_type(emotions):

    positive_score = 0
    negative_score = 0

    for item in emotions:

        emotion = item["emotion"]
        score = item["score"]

        if emotion in positive_emotions:
            positive_score += score

        elif emotion in negative_emotions:
            negative_score += score

    if positive_score > negative_score:
        return "positive"

    elif negative_score > positive_score:
        return "negative"

    return "neutral"

In [ ]:
def analyze_emotion(text, threshold=0.10):

    prediction = emotion_classifier(text)

    formatted_predictions = []

    for item in prediction[0]:

        label_id = int(item["label"].split("_")[1])

        emotion = id2label[label_id]

        score = round(item["score"], 4)

        if score >= threshold:

            formatted_predictions.append({
                "emotion": emotion,
                "score": score
            })

    formatted_predictions = sorted(
        formatted_predictions,
        key=lambda x: x["score"],
        reverse=True
    )

    dominant_emotion = (
        formatted_predictions[0]["emotion"]
        if formatted_predictions
        else "neutral"
    )

    mood_type = get_mood_type(
        formatted_predictions
    )

    return {
        "text": text,
        "mood_type": mood_type,
        "dominant_emotion": dominant_emotion,
        "detected_emotions": formatted_predictions
    }

In [ ]:
result = analyze_emotion(
    "I am extremely stressed and anxious about my future."
)

result

In [ ]:
from keybert import KeyBERT

In [ ]:
keyword_model = KeyBERT()

In [ ]:
def clean_keywords(keywords):

    cleaned = []

    seen = set()

    for item in keywords:

        keyword = item["keyword"].lower().strip()

        words = keyword.split()

        # Remove duplicate-word phrases
        if len(words) != len(set(words)):
            continue

        # Remove repeated keywords
        if keyword in seen:
            continue

        # Ignore very short phrases
        if len(keyword) < 3:
            continue

        cleaned.append({
            "keyword": keyword,
            "score": item["score"]
        })

        seen.add(keyword)

    return cleaned


def extract_keywords(text, top_n=5):

    keywords = keyword_model.extract_keywords(

        text,

        keyphrase_ngram_range=(2, 2),

        stop_words="english",

        top_n=top_n,

        use_mmr=True,

        diversity=0.8
    )

    formatted_keywords = []

    for keyword, score in keywords:

        formatted_keywords.append({
            "keyword": keyword,
            "score": round(score, 4)
        })

    formatted_keywords = clean_keywords(
        formatted_keywords
    )

    return formatted_keywords

In [ ]:
text = """
I feel mentally exhausted because of continuous work pressure and deadlines.
"""

keywords = extract_keywords(text)

print(keywords)

In [ ]:
trigger_categories = {

    "academic stress": [
        "exam",
        "exams",
        "assignment",
        "study",
        "college",
        "university",
        "academic",
        "semester",
        "marks"
    ],

    "work stress": [
        "job",
        "office",
        "manager",
        "meeting",
        "salary",
        "work pressure",
        "coworker",
        "client"
    ],

    "relationship issues": [
        "breakup",
        "relationship",
        "partner",
        "girlfriend",
        "boyfriend",
        "family"
    ],

    "financial stress": [
        "money",
        "rent",
        "loan",
        "debt",
        "expenses",
        "bills"
    ],

    "health anxiety": [
        "health",
        "disease",
        "hospital",
        "pain",
        "illness",
        "doctor"
    ]
}

In [ ]:
def detect_trigger(keywords):

    detected_triggers = []

    keyword_texts = [
        item["keyword"].lower()
        for item in keywords
    ]

    for trigger, trigger_keywords in trigger_categories.items():

        for word in trigger_keywords:

            for keyword in keyword_texts:

                if word in keyword:

                    detected_triggers.append(trigger)

    detected_triggers = list(
        set(detected_triggers)
    )

    return detected_triggers

In [ ]:
keywords = extract_keywords(
    "I am stressed because of exams and assignment deadlines."
)

triggers = detect_trigger(keywords)

print(keywords)
print(triggers)


In [ ]:
from transformers import pipeline

In [ ]:
summary_generator = pipeline(
    "text-generation",
    model="google/flan-t5-small"
)

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

summary_tokenizer = AutoTokenizer.from_pretrained(
    "google/flan-t5-small"
)

summary_model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-small"
)

In [ ]:
def build_summary_prompt(analysis_result):

    dominant_emotion = analysis_result[
        "dominant_emotion"
    ]

    mood_type = analysis_result[
        "mood_type"
    ]

    triggers = ", ".join(
        analysis_result["triggers"]
    )

    keywords = ", ".join([
        item["keyword"]
        for item in analysis_result["keywords"]
    ])

    prompt = f"""
    Generate a short mental health insight summary.

    Mood Type: {mood_type}

    Dominant Emotion: {dominant_emotion}

    Triggers: {triggers}

    Keywords: {keywords}

    Keep the response empathetic, concise, and professional.
    """

    return prompt

In [ ]:
def generate_summary(analysis_result):

    dominant_emotion = analysis_result[
        "dominant_emotion"
    ]

    mood_type = analysis_result[
        "mood_type"
    ]

    triggers = analysis_result[
        "triggers"
    ]

    keywords = [
        item["keyword"]
        for item in analysis_result["keywords"][:3]
    ]

    summary_parts = []

    # Emotion insight
    summary_parts.append(
        f"The entry reflects a {mood_type} emotional state with dominant feelings of {dominant_emotion}."
    )

    # Trigger insight
    if triggers:

        trigger_text = ", ".join(triggers)

        summary_parts.append(
            f"Possible stress triggers include {trigger_text}."
        )

    # Keyword insight
    if keywords:

        keyword_text = ", ".join(keywords)

        summary_parts.append(
            f"Key emotional indicators include {keyword_text}."
        )

    return " ".join(summary_parts)

In [ ]:
def analyze_mental_health(text):

    emotion_result = analyze_emotion(text)

    keywords = extract_keywords(text)

    triggers = detect_trigger(keywords)

    analysis_result = {

        "text": text,

        "mood_type":
            emotion_result["mood_type"],

        "dominant_emotion":
            emotion_result["dominant_emotion"],

        "detected_emotions":
            emotion_result["detected_emotions"],

        "keywords":
            keywords,

        "triggers":
            triggers
    }

    analysis_result["summary"] = (
        generate_summary(analysis_result)
    )

    return analysis_result

In [ ]:
def analyze_mental_health(text):

    emotion_result = analyze_emotion(text)

    keywords = extract_keywords(text)

    triggers = detect_trigger(keywords)

    analysis_result = {

        "text": text,

        "mood_type":
            emotion_result["mood_type"],

        "dominant_emotion":
            emotion_result["dominant_emotion"],

        "detected_emotions":
            emotion_result["detected_emotions"],

        "keywords":
            keywords,

        "triggers":
            triggers
    }

    analysis_result["summary"] = (
        generate_summary(analysis_result)
    )

    return analysis_result

In [ ]:
def analyze_mental_health(text):

    emotion_result = analyze_emotion(text)

    keywords = extract_keywords(text)

    triggers = detect_trigger(keywords)

    analysis_result = {

        "text": text,

        "mood_type":
            emotion_result["mood_type"],

        "dominant_emotion":
            emotion_result["dominant_emotion"],

        "detected_emotions":
            emotion_result["detected_emotions"],

        "keywords":
            keywords,

        "triggers":
            triggers
    }

    analysis_result["summary"] = (
        generate_summary(analysis_result)
    )

    return analysis_result

In [ ]:
result = analyze_mental_health(
    """
    I feel lonely and emotionally exhausted because
    of continuous work pressure and deadlines.
    """
)

print(result)


In [ ]:
weekly_entries = [

    {
        "day": "Monday",
        "text": "I feel stressed because of exams."
    },

    {
        "day": "Tuesday",
        "text": "I am mentally exhausted from assignments."
    },

    {
        "day": "Wednesday",
        "text": "I feel anxious about my future and career."
    },

    {
        "day": "Thursday",
        "text": "Today was peaceful and relaxing."
    },

    {
        "day": "Friday",
        "text": "I feel lonely and emotionally drained."
    }
]

In [ ]:
weekly_analysis = []

for entry in weekly_entries:

    result = analyze_mental_health(
        entry["text"]
    )

    result["day"] = entry["day"]

    weekly_analysis.append(result)

In [ ]:
for item in weekly_analysis:

    print(item["day"])

    print(item["dominant_emotion"])

    print(item["mood_type"])

    print("-" * 50)

In [ ]:
from collections import Counter

In [ ]:
emotion_counter = Counter()

for item in weekly_analysis:

    emotion_counter[
        item["dominant_emotion"]
    ] += 1

print(emotion_counter)

In [ ]:
mood_counter = Counter()

for item in weekly_analysis:

    mood_counter[
        item["mood_type"]
    ] += 1

print(mood_counter)

In [ ]:
most_common_emotion = emotion_counter.most_common(1)[0]

print(most_common_emotion)

In [ ]:
def generate_weekly_insight(

    emotion_counter,

    mood_counter

):

    dominant_emotion = (
        emotion_counter.most_common(1)[0][0]
    )

    dominant_mood = (
        mood_counter.most_common(1)[0][0]
    )

    summary = f"""
    Weekly emotional analysis indicates a predominantly
    {dominant_mood} mood pattern.

    The most frequently detected emotion was
    {dominant_emotion}.

    Continued monitoring may help identify emotional
    triggers and behavioral trends over time.
    """

    return summary.strip()

In [ ]:
weekly_summary = generate_weekly_insight(

    emotion_counter,

    mood_counter
)

print(weekly_summary)

In [ ]:
trend_data = []

for item in weekly_analysis:

    trend_data.append({

        "day": item["day"],

        "emotion": item["dominant_emotion"],

        "mood": item["mood_type"]

    })

trend_data

In [ ]:
emotion_scores = []

for item in weekly_analysis:

    dominant_score = item[
        "detected_emotions"
    ][0]["score"]

    emotion_scores.append({

        "day": item["day"],

        "emotion": item[
            "dominant_emotion"
        ],

        "score": dominant_score
    })

emotion_scores

In [ ]:
def generate_advanced_weekly_insight(
    weekly_analysis
):

    negative_days = 0
    positive_days = 0

    dominant_emotions = []

    triggers = []

    for item in weekly_analysis:

        dominant_emotions.append(
            item["dominant_emotion"]
        )

        triggers.extend(
            item["triggers"]
        )

        if item["mood_type"] == "negative":
            negative_days += 1

        elif item["mood_type"] == "positive":
            positive_days += 1

    emotion_counter = Counter(
        dominant_emotions
    )

    trigger_counter = Counter(
        triggers
    )

    most_common_emotion = (
        emotion_counter.most_common(1)[0][0]
    )

    most_common_trigger = (
        trigger_counter.most_common(1)[0][0]
        if trigger_counter
        else "unknown"
    )

    summary = f"""
    Weekly emotional trends indicate
    {negative_days} negative emotional days and
    {positive_days} positive emotional days.

    The most common emotional pattern was
    {most_common_emotion}.

    The most frequent trigger category was
    {most_common_trigger}.

    Monitoring these trends over time may help
    identify recurring emotional stress patterns.
    """

    return summary.strip()

In [ ]:
def generate_advanced_weekly_insight(
    weekly_analysis
):

    negative_days = 0
    positive_days = 0

    dominant_emotions = []

    triggers = []

    for item in weekly_analysis:

        dominant_emotions.append(
            item["dominant_emotion"]
        )

        triggers.extend(
            item["triggers"]
        )

        if item["mood_type"] == "negative":
            negative_days += 1

        elif item["mood_type"] == "positive":
            positive_days += 1

    emotion_counter = Counter(
        dominant_emotions
    )

    trigger_counter = Counter(
        triggers
    )

    most_common_emotion = (
        emotion_counter.most_common(1)[0][0]
    )

    most_common_trigger = (
        trigger_counter.most_common(1)[0][0]
        if trigger_counter
        else "unknown"
    )

    summary = f"""
    Weekly emotional trends indicate
    {negative_days} negative emotional days and
    {positive_days} positive emotional days.

    The most common emotional pattern was
    {most_common_emotion}.

    The most frequent trigger category was
    {most_common_trigger}.

    Monitoring these trends over time may help
    identify recurring emotional stress patterns.
    """

    return summary.strip()

In [ ]:
def generate_advanced_weekly_insight(
    weekly_analysis
):

    negative_days = 0
    positive_days = 0

    dominant_emotions = []

    triggers = []

    for item in weekly_analysis:

        dominant_emotions.append(
            item["dominant_emotion"]
        )

        triggers.extend(
            item["triggers"]
        )

        if item["mood_type"] == "negative":
            negative_days += 1

        elif item["mood_type"] == "positive":
            positive_days += 1

    emotion_counter = Counter(
        dominant_emotions
    )

    trigger_counter = Counter(
        triggers
    )

    most_common_emotion = (
        emotion_counter.most_common(1)[0][0]
    )

    most_common_trigger = (
        trigger_counter.most_common(1)[0][0]
        if trigger_counter
        else "unknown"
    )

    summary = f"""
    Weekly emotional trends indicate
    {negative_days} negative emotional days and
    {positive_days} positive emotional days.

    The most common emotional pattern was
    {most_common_emotion}.

    The most frequent trigger category was
    {most_common_trigger}.

    Monitoring these trends over time may help
    identify recurring emotional stress patterns.
    """

    return summary.strip()

In [ ]:
advanced_summary = generate_advanced_weekly_insight(
    weekly_analysis
)

print(advanced_summary)

In [ ]:
trainer.save_model(
    "/content/drive/MyDrive/MentalHealthAI/final_emotion_model"
)

tokenizer.save_pretrained(
    "/content/drive/MyDrive/MentalHealthAI/final_emotion_model"
)

print("Model saved successfully.")

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio

In [ ]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3EQvg4k2MW4Z0ltS4dNXgPE6FNl_7CtTFQHTRsVFb6i7G1WJo"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [ ]:
result = analyze_mental_health(
    "I feel stressed because of exams."
)

print(result)